In [ ]:
# 数据处理库
import sys
import json
import logging
import pandas as pd
import numpy as np
from dataset_deduplicate_utils import*
from datetime import datetime

In [ ]:
logging.basicConfig(
    level=logging.INFO,  # 强制要求显示 INFO 级别的普通信息
    format='%(asctime)s [%(levelname)s] %(message)s', # 设置日志长什么样（带上时间和级别）
    handlers=[
        logging.StreamHandler(sys.stdout) # 确保日志打印在屏幕上
    ]
)

In [ ]:
class Config:
    
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    IN_DIR = '../data/input'
    IN_OUT_DIR = '../data/in_out'
    OUT_DIR = '../data/output'
    TEMP_DIR = '../tmp_data'

    config_path = '../configs/pretreatment_config.json'

    try:
        with open(config_path, 'r', encoding='utf-8') as f:
            _raw_config = json.load(f)
            _raw_dataset_name = _raw_config.get("RAW_DATASET_NAME")
            _property = _raw_config.get("PROPERTY")
            _state_var = _raw_config.get("STATE_VAR", "")
            _reference = _raw_config.get("REFERENCE")
            
    except FileNotFoundError:
        logging.warning(f"配置文件未找到: {config_path}")    


    _base_name = os.path.splitext(_raw_dataset_name)[0]

    MISSING_DATA_MAG_DIR = os.path.join(TEMP_DIR, _base_name, 'mis_data_mag')
    DEDUPLICATE_DIR = os.path.join(TEMP_DIR, _base_name, 'deduplicate')

    # 输入文件路径
    DATADET_WITH_IDS = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_raw-ds_add-ids_dd.xlsx')

    # 输出文件路径
    DATASET_DEDUPLICATED = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_ds-with-ids_dd_fg.xlsx')
    DATASET_DEDUPLICATED_OUT = os.path.join(OUT_DIR, f'{_base_name}_{TIMESTAMP}_out_ds-dd.xlsx')

In [ ]:
TIMESTAMP = datetime.now().strftime('%y%m%d') 

# 【解耦修改】从 Config 类动态读取配置变量
property_col = Config._property
ref_col = Config._reference
state_var_col = Config._state_var

# 【解耦修改】动态构建 essential_cols
essential_cols = ['c1', 'c2', 'x1', 'x2']
if property_col: essential_cols.append(property_col)
if ref_col: essential_cols.append(ref_col)
if state_var_col: essential_cols.append(state_var_col)

# 【配置】 定义组分名称列 (使用 SMILES 作为唯一标识)
# 注意：这三列将用于生成指纹，并在排序中起决定性作用
str_cols = [
    'c1_ismiles',
    'c2_ismiles',
    'c3_ismiles'
]

# 【配置】 定义组分比例列
# 注意：这里只包含比例 (x)，不能包含温度 (t)
num_cols = [
    'x1',
    'x2', 
    'x3'
]

# 中间结果输出目录
interim_output_dir = Config.DEDUPLICATE_DIR

# 定义处理完后要删除的辅助列 (例如原始的 c1, c2 名称列，如果不再需要)
drop_columns = ['c1', 'c2', 'c3']

In [ ]:
# 1. 读取文件
print("\n正在加载原始数据集...")
original_df = pd.read_excel(Config.DATADET_WITH_IDS)
# =========================================================
# 2. 【阶段一】调用 utils 中的体检与重复项打标函数
# =========================================================
print("\n>>> 启动阶段一：识别重复数据 <<<")
intermediate_df = identify_and_export_duplicates(
    original_df=original_df,
    essential_cols=essential_cols,
    str_cols=str_cols,       # 传入组分标识列
    num_cols=num_cols,       # 传入比例列
    output_dir=interim_output_dir,
    ref_col=ref_col,         # 动态参考文献列名
    state_var_cols=state_var_col   
)

# 安全拦截：如果数据在阶段一被全部洗掉了，直接终止
if intermediate_df is None or intermediate_df.empty:
    print("❌ 数据清洗后无有效数据，程序终止。请检查输入数据质量。")
    import sys
    sys.exit(1)

In [ ]:
# 如果想用中位数抵抗离群值：
result_df = fuse_and_assemble_dataset(
    data_df=intermediate_df,
    property_col=property_col,
    drop_columns=drop_columns,
    ref_col=ref_col,
    agg_method='mean'
)

result_df.to_excel(Config.DATASET_DEDUPLICATED_OUT)